<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/weekend-05-rag-research/carlos-lectures/reranking/exercises/reranking_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reranking: how to judge the 100

**From Data to Solutions, Weekend 5.** Coding exercises for the reranking block.

A search engine hands your assistant the 100 paragraphs it thinks are relevant.
A reranker reads those 100 properly and hands back the best 5, best one on top.
In this notebook you build three of them, the three from the lecture:

| | What you build | The idea in one line | Time |
|---|---|---|---|
| **1** | **monoBERT** | read the question and the paragraph together, score, sort | 20 min |
| **2** | **ColBERT** | one vector per word, compare word to word, and train it | 30 min |
| **3** | **RankGPT** | ask a language model for the order, through a sliding window | 25 min |

### How to use this notebook

* Run every cell from the top, in order.
* **A 🎯 marks a gap you fill in.** There are 11 of them. Everything else runs by itself.
* Each gap is one or two lines. The comment right above it tells you exactly what to write.
* After each exercise there is a check cell that tells you whether it worked, and gives a
  hint when it did not.
* Cells that say *(run me, no need to read)* are plumbing. They are folded away on purpose.

Carlos Cotrini


In [ ]:
# @title Install the two packages we need (about one minute)
!pip install -q sentence-transformers google-genai
print("done")


In [ ]:
# @title The company archive, the fast search, and the helper functions (run me, no need to read) { display-mode: "form" }
DOCS = [
    dict(id=0, source="Nordic Supply Agreement, section 7",
         text="Nordic Supply Agreement, section 7. Alpine Instruments shall deliver all standard catalogue items to the customer's Nordic warehouses within 5 working days of order confirmation. Custom built instruments are delivered within 20 working days."),
    dict(id=1, source="Nordic Supply Agreement, section 9",
         text="Nordic Supply Agreement, section 9. The customer shall settle every invoice issued under this agreement within 30 days of the invoice date. Late payment accrues interest at 4 percent per year."),
    dict(id=2, source="Baltic Supply Agreement, section 7",
         text="Baltic Supply Agreement, section 7. Alpine Instruments shall deliver all standard catalogue items to the customer's Baltic warehouses within 15 working days of order confirmation. Custom built instruments are delivered within 40 working days."),
    dict(id=3, source="Warehouse Handbook, chapter 4",
         text="Warehouse Handbook, chapter 4. Typical shipment lead time from the Zurich warehouse to a northern European destination is 3 to 6 days, depending on the carrier and on customs clearance. These are planning figures and not commitments to any customer."),
    dict(id=4, source="Baltic Supply Agreement, section 12",
         text="Baltic Supply Agreement, section 12. If a delivery is late by more than 10 working days, the customer may claim a penalty of 2 percent of the order value per started week, capped at 10 percent of the order value."),
    dict(id=5, source="Iberian Supply Agreement, section 12",
         text="Iberian Supply Agreement, section 12. If a delivery is late by more than 10 working days, the customer may claim a penalty of 3 percent of the order value per started week, capped at 15 percent of the order value."),
    dict(id=6, source="Employee Handbook, chapter 2",
         text="Employee Handbook, chapter 2. Employees on a full time contract receive 25 days of paid annual leave. Employees over the age of 50 receive 27 days. Unused leave may be carried into the first quarter of the following year."),
    dict(id=7, source="Employee Handbook, chapter 3",
         text="Employee Handbook, chapter 3. Parental leave is 16 weeks for the birthing parent and 4 weeks for the second parent, paid at 80 percent of salary. It must be requested 12 weeks in advance."),
    dict(id=8, source="Procurement Policy, section 5",
         text="Procurement Policy, section 5. Alpine Instruments pays its own suppliers 45 days after the goods are received and the invoice has been approved. Suppliers rated strategic may negotiate 30 days."),
    dict(id=9, source="Finance Manual, section 8",
         text="Finance Manual, section 8. Customer invoices are issued on the day of shipment. The standard payment window granted to a customer is 30 days, and the credit team reviews any request to extend it."),
    dict(id=10, source="Sales Approval Matrix",
         text="Sales Approval Matrix. A discount up to 10 percent may be granted by the account manager. Above 10 percent and up to 15 percent it needs the regional sales director. Any discount above 15 percent requires the chief financial officer in writing."),
    dict(id=11, source="Sales Handbook, chapter 6",
         text="Sales Handbook, chapter 6. When a customer asks for a price reduction, first offer a longer warranty or free installation. Discounts are the last instrument, because they are almost never reversed at the next renewal."),
    dict(id=12, source="Quarterly Report Q3, Nordics",
         text="Quarterly Report Q3, Nordics. Revenue in the Nordic region was 4.2 million francs in the third quarter, up 12 percent on the same quarter last year. Growth came mainly from service contracts rather than from instrument sales."),
    dict(id=13, source="Quarterly Report Q2, Nordics",
         text="Quarterly Report Q2, Nordics. Revenue in the Nordic region was 3.6 million francs in the second quarter, flat against the same quarter last year. Two large instrument orders slipped into the following quarter."),
    dict(id=14, source="Board Minutes, March",
         text="Board Minutes, March. The board heard repeated complaints from customers in the north and asked operations to report back in June on whether the windows written into the supply agreements are being met in practice."),
    dict(id=15, source="Quality Manual, section 2",
         text="Quality Manual, section 2. Every instrument is calibrated before shipment and travels with a calibration certificate valid for 12 months. A recalibration service is offered at the customer site."),
    dict(id=16, source="IT Policy, section 3",
         text="IT Policy, section 3. Company laptops are encrypted and are replaced every four years. Personal cloud storage may not be used for customer data of any kind."),
    dict(id=17, source="Travel Policy, section 1",
         text="Travel Policy, section 1. Train is the default for any journey under six hours. A flight needs line manager approval, and economy class is the standard for any flight under five hours."),
    dict(id=18, source="Facilities Notice",
         text="Facilities Notice. The staff cafeteria opens at 07:00 and serves a hot lunch from 11:30 to 13:30. The building is closed on Sundays except for badge holders in the laboratory wing."),
    dict(id=19, source="Environmental Report, section 4",
         text="Environmental Report, section 4. Freight emissions fell 8 percent after the switch from air freight to rail for shipments inside Europe. The remaining air freight is used only for urgent spare parts."),
    dict(id=20, source="Employee Handbook, chapter 5",
         text="Employee Handbook, chapter 5. Public holidays follow the canton of Zurich, which recognises 9 public holidays in the year. A public holiday that falls on a weekend is not moved to a working day."),
    dict(id=21, source="Treasury Note, payment runs",
         text="Treasury Note, payment runs. Supplier payments leave the company twice a month, on the 10th and on the 25th. An invoice approved after the cut off waits for the next run."),
    dict(id=22, source="Delegation of Authority, section 2",
         text="Delegation of Authority, section 2. Any purchase commitment above 50,000 francs must be signed by two members of the executive board. Below that threshold the department head signs alone."),
    dict(id=23, source="Service Contract Terms, section 3",
         text="Service Contract Terms, section 3. An on site repair is started within 2 working days of a fault report during the warranty period. Outside the warranty the response target is 5 working days."),
    dict(id=24, source="Quarterly Report Q3, Iberia",
         text="Quarterly Report Q3, Iberia. Revenue in the Iberian region was 1.8 million francs in the third quarter, down 4 percent on the same quarter last year, after two service contracts were not renewed."),
    dict(id=25, source="Onboarding Guide, week one",
         text="Onboarding Guide, week one. A new colleague receives a laptop and a badge on the first morning, meets the safety officer before entering the laboratory, and is assigned a buddy for the first three months."),
    dict(id=26, source="Warranty Terms, section 1",
         text="Warranty Terms, section 1. Every instrument carries a 24 month warranty from the date of delivery. The warranty covers parts and labour but not consumables or damage from incorrect use."),
    dict(id=27, source="Expense Rules, section 4",
         text="Expense Rules, section 4. Meals during a business trip are reimbursed up to 60 francs per day inside Switzerland and up to 80 francs abroad. Receipts must be submitted within 30 days."),
]

# gold: the one paragraph that actually answers the question.
QUERIES = [
    dict(qid=1,  text="What did we promise on delivery times in the Nordic contract?", gold=0),
    dict(qid=2,  text="How many days of paid holiday does a full time employee get?", gold=6),
    dict(qid=3,  text="What penalty applies if a Baltic delivery is late?", gold=4),
    dict(qid=4,  text="How quickly do we pay our own suppliers?", gold=8),
    dict(qid=5,  text="Who has to sign off a discount of 20 percent?", gold=10),
    dict(qid=6,  text="What was revenue in the Nordics in the third quarter?", gold=12),
    dict(qid=7,  text="How long is the warranty on an instrument?", gold=26),
    dict(qid=8,  text="How fast do we start a repair while the instrument is still under warranty?", gold=23),
    dict(qid=9,  text="How much can I spend on dinner on a trip abroad?", gold=27),
    dict(qid=10, text="When do we deliver a custom built instrument to a Baltic customer?", gold=2),
    dict(qid=11, text="How long does a customer have to pay an invoice?", gold=9),
    dict(qid=12, text="Do I need approval to fly to a meeting?", gold=17),
]

# The same eight questions as different colleagues typed them.
# The held out questions (1, 6, 8, 12) deliberately have no paraphrases.
PARAPHRASES = {
    2: ["How much annual leave do I get on a full time contract?",
        "Number of vacation days per year for a full timer?",
        "paid holiday entitlement full time staff"],
    3: ["If we ship late to a Baltic customer, what do we owe them?",
        "Baltic agreement, what is the late delivery penalty?",
        "compensation for a delayed shipment under the Baltic contract"],
    4: ["What is our payment term towards suppliers?",
        "After how many days does a supplier invoice get paid?",
        "supplier payment terms"],
    5: ["Which manager approves a 25 percent discount?",
        "Do I need the CFO for a discount above 15 percent?",
        "discount approval authority levels"],
    7: ["What is the warranty period of our instruments?",
        "How many months of warranty do customers get?",
        "instrument warranty duration"],
    9: ["Meal allowance for a business trip outside Switzerland?",
        "What is the daily food budget abroad?",
        "reimbursement limit for dinner while travelling abroad"],
    10: ["Lead time for a bespoke instrument for a Baltic client?",
         "How long does a custom build take under the Baltic agreement?",
         "Baltic contract delivery window for made to order units"],
    11: ["What payment window does a customer get on an invoice?",
         "How many days does a client have to settle an invoice?",
         "customer invoice due date policy"],
}

TRAIN_QIDS = [2, 3, 4, 5, 7, 9, 10, 11]
TEST_QIDS = [1, 6, 8, 12]

import time, random, re, math, textwrap
import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

random.seed(0); np.random.seed(0); torch.manual_seed(0)

ANCHOR = QUERIES[0]["text"]
BY_QID = {q["qid"]: q for q in QUERIES}
TEXTS = [d["text"] for d in DOCS]


def gold_of(query):
    """The one paragraph that really answers this question."""
    for q in QUERIES:
        if q["text"] == query:
            return q["gold"]
    return None


print("loading the search model (about 90 MB, once) ...")
_bi = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_INDEX = _bi.encode(TEXTS, normalize_embeddings=True, show_progress_bar=False)


def first_stage(query, k=10):
    """Stage one: the fast search. One vector per paragraph, cosine similarity.
    Returns the ids of the k best paragraphs, best first."""
    qv = _bi.encode(query, normalize_embeddings=True, show_progress_bar=False)
    return [int(i) for i in np.argsort(-(_INDEX @ qv))[:k]]


def show_archive():
    print("The archive: %d paragraphs\n" % len(DOCS))
    for d in DOCS:
        print("  [%2d]  %-38s %s..." % (d["id"], d["source"], d["text"][len(d["source"]) + 2:][:58]))


def _wrapped(text, indent="    ", width=88):
    return textwrap.indent(textwrap.fill(text, width), indent)


def _dropdown(options, value, label):
    from ipywidgets import Dropdown, Layout
    return Dropdown(options=options, value=value, description=label,
                    style={"description_width": "initial"}, layout=Layout(width="88%"))


def explore_documents():
    """Browse the archive, one paragraph at a time."""
    answers = {q["gold"]: q for q in QUERIES}

    def view(paragraph):
        d = DOCS[paragraph]
        print("[%d]   %s\n" % (d["id"], d["source"]))
        print(_wrapped(d["text"]))
        if d["id"] in answers:
            print("\n  This is the paragraph that answers question %d:"
                  % answers[d["id"]]["qid"])
            print('    "%s"' % answers[d["id"]]["text"])
        else:
            print("\n  No question in our test set is answered by this paragraph.")

    try:
        from ipywidgets import interact
        interact(view, paragraph=_dropdown(
            [("[%2d]  %s" % (d["id"], d["source"]), d["id"]) for d in DOCS], 0, "paragraph:"))
    except Exception:
        view(0)
        print("\n(the dropdown needs ipywidgets. Call view_document(i) instead.)")


def view_document(i):
    d = DOCS[i]
    print("[%d]   %s\n" % (d["id"], d["source"]))
    print(_wrapped(d["text"]))


def explore_questions():
    """Browse the questions: what each one asks, what answers it, and what gets in the way."""
    def view(question):
        q = BY_QID[question]
        print("THE QUESTION")
        print('    "%s"\n' % q["text"])
        g = DOCS[q["gold"]]
        print("THE PARAGRAPH THAT ANSWERS IT")
        print("    [%d]  %s" % (g["id"], g["source"]))
        print(_wrapped(g["text"], "      "))
        print("\nPARAGRAPHS ABOUT THE SAME SUBJECT THAT DO NOT ANSWER IT")
        print("(this is what makes the question hard: same words, wrong paragraph)\n")
        shown = 0
        for i in first_stage(q["text"], 6):
            if i == q["gold"] or shown >= 3:
                continue
            print("    [%d]  %s" % (i, DOCS[i]["source"]))
            print(_wrapped(DOCS[i]["text"], "      "))
            print()
            shown += 1

    try:
        from ipywidgets import interact
        interact(view, question=_dropdown(
            [("Q%d   %s" % (q["qid"], q["text"]), q["qid"]) for q in QUERIES], 1, "question:"))
    except Exception:
        view(1)
        print("\n(the dropdown needs ipywidgets.)")


def show_ranking(query, ids, k=5, title=""):
    """Print a ranking, marking the paragraph that actually answers the question."""
    g = gold_of(query)
    if title:
        print(title)
    print('  question: "%s"' % query)
    for rank, i in enumerate(ids[:k], 1):
        mark = "  <== the answer" if i == g else ""
        print("   %d.  [%2d]  %-40s%s" % (rank, i, DOCS[i]["source"], mark))
    if g is not None and g not in ids[:k]:
        pos = ids.index(g) + 1 if g in ids else None
        print("   ...  the answer is at position %s" % (pos if pos else "not in the list"))
    print()


def evaluate(rank_fn, label, qids=None, depth=10, quiet=False):
    """Run a reranker over the question set and score it.

    rank_fn(query, doc_ids) must return the SAME ids, reordered, best first.
    Reports accuracy at 1 (how often the true answer ends up on top) and MRR
    (1 if the answer is first, 1/2 if second, 1/3 if third, and so on)."""
    qids = qids or [q["qid"] for q in QUERIES]
    hits, mrr, rows = 0, 0.0, []
    for qid in qids:
        q = BY_QID[qid]
        cand = first_stage(q["text"], depth)
        order = rank_fn(q["text"], cand) if rank_fn else cand
        if sorted(order) != sorted(cand):
            raise ValueError("the reranker must return the same ids it was given, only reordered")
        r = order.index(q["gold"]) + 1
        rows.append((qid, q["text"], r))
        hits += (r == 1)
        mrr += 1.0 / r
    n = len(qids)
    if not quiet:
        print("%s\n" % label)
        print("      question                                                        answer ends up at")
        for qid, text, r in rows:
            flag = "" if r == 1 else "   <-- not first"
            print("  %2d  %-62s %2d%s" % (qid, text[:62], r, flag))
        print("\n  accuracy at 1:  %d/%d        MRR:  %.3f\n" % (hits, n, mrr / n))
    return {"acc": hits, "n": n, "mrr": mrr / n, "ranks": [r for _, _, r in rows]}


def recall_at(depth):
    """How often stage one puts the answer somewhere in its top `depth`."""
    return sum(q["gold"] in first_stage(q["text"], depth) for q in QUERIES)


def ok(msg):
    print("  ✅ " + msg)


def bad(msg):
    print("  ❌ " + msg)


def hint(msg):
    print("     hint: " + msg)


def report(title, passed):
    print("\n" + ("✅ " if passed else "❌ ") + title + ("" if passed else "  (see the hints above)"))
    return passed


def check_docs(source_of_6, text_of_12, texts_of_3_and_7):
    print("checking your three answers ...\n")
    passed = True
    if source_of_6 != DOCS[6]["source"]:
        bad("a) you got %r" % (source_of_6,))
        hint('DOCS[6] is the whole entry. Add ["source"] to pull one field out of it.')
        passed = False
    else:
        ok("a) the source of paragraph 6")

    if text_of_12 != DOCS[12]["text"]:
        bad("b) you got %r" % (str(text_of_12)[:60],))
        hint('same shape as a), but the field is called "text"')
        passed = False
    else:
        ok("b) the text of paragraph 12")

    want = [DOCS[3]["text"], DOCS[7]["text"]]
    if not isinstance(texts_of_3_and_7, list):
        bad("c) you got a %s, it should be a list" % type(texts_of_3_and_7).__name__)
        hint("the square brackets of a list comprehension build the list for you")
        passed = False
    elif len(texts_of_3_and_7) != 2:
        bad("c) your list has %d entries, it should have 2" % len(texts_of_3_and_7))
        passed = False
    elif texts_of_3_and_7 == [3, 7] or texts_of_3_and_7 == [DOCS[3], DOCS[7]]:
        bad("c) your list does not hold the texts")
        hint('each entry should be DOCS[i]["text"], not i and not DOCS[i]')
        passed = False
    elif texts_of_3_and_7 != want:
        bad("c) the two texts are not the ones from paragraphs 3 and 7")
        passed = False
    else:
        ok("c) a list holding the text of paragraph 3 and the text of paragraph 7")
    return report("Warm up: the archive", passed)


print("ready. %d paragraphs indexed, %d questions." % (len(DOCS), len(QUERIES)))


## The situation

A company has an internal assistant over its own documents: contracts, handbooks, reports,
board minutes. Someone asks it:

> **What did we promise on delivery times in the Nordic contract?**

Here is the whole archive. In a real company this would be a few million paragraphs.
Twenty eight is enough to see everything that matters.


In [ ]:
show_archive()


### Look around before you build anything

Two browsers, so that the data is not a black box. Pick from the dropdown and the cell below it
redraws.

The first one walks the archive paragraph by paragraph. Read three or four. Notice how ordinary
they are, and that several of them are about delivery, several about payment, several about
leave.


In [ ]:
explore_documents()


The second one is the more useful of the two. For each of our twelve test questions it shows the
one paragraph that really answers it, and then the paragraphs that talk about the same subject in
the same words and still do not answer it.

Those near misses are the whole difficulty. Question 1 asks what we promised on delivery times in
the **Nordic** contract, and the archive also contains the Nordic payment clause, the **Baltic**
delivery clause, and a warehouse handbook talking about shipment lead times. Every question in the
set is built this way on purpose.


In [ ]:
explore_questions()


### The archive in Python

You will be handing pieces of this archive to models all morning, so spend one minute on what it
actually is. `DOCS` is a plain Python **list**. Each entry is a **dictionary** with three fields:

| field | what it holds |
|---|---|
| `"id"` | the number in square brackets above, and the position in the list |
| `"source"` | which document the paragraph came from |
| `"text"` | the paragraph itself, the thing a model reads |

So `DOCS[6]` is the whole seventh entry, and `DOCS[6]["text"]` is just its text.


In [ ]:
print("DOCS is a %s with %d entries\n" % (type(DOCS).__name__, len(DOCS)))

print("DOCS[6] is one whole entry:")
print("  ", DOCS[6])

print("\nand you pull one field out of it by name:")
print('  DOCS[6]["id"]      ->', DOCS[6]["id"])
print('  DOCS[6]["source"]  ->', DOCS[6]["source"])
print('  DOCS[6]["text"]    ->', DOCS[6]["text"][:55], "...")

print("\nto collect several texts at once, a list comprehension:")
print("  ", [DOCS[i]["text"][:28] + " ..." for i in [0, 1]])


Your turn. Three one liners, so that the pattern is in your fingers before you need it.


In [ ]:
# 🎯 1  Fill in the three blanks, then run the cell.

# a) the source of paragraph 6
source_of_6 = ...

# b) the full text of paragraph 12
text_of_12 = ...

# c) the texts of paragraphs 3 and 7, as a list of two strings.
#    Use the list comprehension pattern from the cell above:  [ something for i in [3, 7] ]
texts_of_3_and_7 = ...


check_docs(source_of_6, text_of_12, texts_of_3_and_7)


## Stage one: the fast search, and what is wrong with it

The fast search squeezed each paragraph into **one** list of 384 numbers, in advance.
At question time it squeezes the question the same way and looks for the nearest paragraphs.
Millions of paragraphs in milliseconds, which is why every system starts this way.

Look at what it returns for our question.


In [ ]:
show_ranking(ANCHOR, first_stage(ANCHOR, 10), k=5,
             title="Stage one, the fast search:")


The answer is paragraph 0, the Nordic delivery clause. The fast search did find it,
but it put something else first. One squeezed vector per paragraph cannot tell
"we deliver within 5 working days" from "the customer shall settle every invoice within 30 days"
when both come from the same contract.

Now the same measurement over all twelve questions in our test set.


In [ ]:
base = evaluate(None, "Stage one alone (no reranking):")
print("the answer is somewhere in the top 10 for %d of 12 questions" % recall_at(10))


Read those two numbers together, because they are the whole reason reranking exists.

* The fast search puts the answer **first** for 8 of 12 questions.
* The fast search has the answer **somewhere in its top 10** for **12 of 12** questions.

Nothing was lost. The answer is in the stack every single time, just not on top.
A second, slower pass over those 10 paragraphs can fix the order, and it can never
do better than that 12 of 12: **stage two can only reorder what stage one found.**

That is the job. Three ways to do it, cheapest first.


---
# Exercise 1. monoBERT: read the question and the paragraph together

The fast search read the question and the paragraph **separately**: it squeezed each into its
own vector, months apart, and then compared the two vectors. The words of the question never
met the words of the paragraph.

A **cross encoder** does the opposite. It glues question and paragraph into one piece of text
and reads them together, in one go:

```
[CLS] what did we promise on delivery times in the Nordic contract [SEP]
      Nordic Supply Agreement, section 7. Alpine Instruments shall deliver ... [SEP]
```

Now every word of the question can look at every word of the paragraph. The model was trained
to answer one question about that glued text: *is this paragraph relevant to that query, yes or
no?* We keep the model's confidence in "yes" and use it as the score.

That is monoBERT. The paper is called "Passage Re-ranking with BERT" (Nogueira and Cho, 2019),
and it never uses the name monoBERT; the name came later, to tell it apart from the pairwise
version. The model we load below is a small public cross encoder trained exactly this way.

**The price:** the paragraph cannot be prepared in advance. Nothing can be computed until the
question arrives, and then the model has to run once per candidate. That is why it never runs
on the whole archive, only on the shortlist stage one produced.


In [ ]:
# @title Load the cross encoder and the checkers (run me, no need to read) { display-mode: "form" }
from sentence_transformers import CrossEncoder

print("loading the cross encoder (about 90 MB, once) ...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def check_one_score(score):
    print("checking your score ...\n")
    q = "What did we promise on delivery times in the Nordic contract?"
    want = float(cross_encoder.predict([(q, DOCS[0]["text"])])[0])
    try:
        got = float(score)
    except Exception:
        kind = type(score).__name__
        if kind in ("ellipsis", "NoneType"):
            bad("the gap is still empty")
        else:
            bad("your answer is a %s, it should be one number" % kind)
            hint("predict gives back a list of scores, one per pair. Take the first one with [0].")
        return report("Warm up: the cross encoder", False)
    if abs(got - want) > 1e-3:
        bad("you got %.3f, the right answer is %.3f" % (got, want))
        hint("the pair has to be (the question, the TEXT of paragraph 0), in that order")
        return report("Warm up: the cross encoder", False)
    ok("%.3f, which is the same score the cell above printed for the right paragraph" % got)
    return report("Warm up: the cross encoder", True)


def check_monobert(fn):
    print("checking your rerank_monobert ...\n")
    q = "What penalty applies if a Baltic delivery is late?"
    cand = first_stage(q, 6)
    try:
        out = fn(q, cand)
    except Exception as e:
        bad("your function raised %s: %s" % (type(e).__name__, e))
        hint("one of the 🎯 gaps is probably still a bare `...`. Every `...` has to go.")
        return report("monoBERT", False)

    passed = True
    if not isinstance(out, (list, tuple)):
        bad("the function returned a %s, it should return a list of ids" % type(out).__name__)
        return report("monoBERT", False)
    out = list(out)
    if sorted(out) != sorted(cand):
        bad("you must return the same %d ids you were given, only in a different order" % len(cand))
        hint("you were given %s and returned %s" % (sorted(cand), sorted(out)))
        return report("monoBERT", False)
    ok("you return the same paragraphs, reordered")

    sc = [float(s) for s in cross_encoder.predict([(q, DOCS[i]["text"]) for i in out])]
    if all(sc[i] <= sc[i + 1] for i in range(len(sc) - 1)) and len(set(sc)) > 1:
        bad("your ranking is upside down: the worst paragraph comes first")
        hint("check that the pairs in 🎯 3 are (question, text) and not (text, question)")
        passed = False
    elif not all(sc[i] >= sc[i + 1] for i in range(len(sc) - 1)):
        bad("the ids are not sorted by score")
        hint("scores in the order you returned: %s" % [round(s, 2) for s in sc])
        hint("the pairs in 🎯 3 must come in the same order as doc_ids, so that scores[k] "
             "really is the score of doc_ids[k]")
        passed = False
    else:
        ok("sorted by score, largest first")

    known = [3, 5, 12]
    wrong = []
    for qid in known:
        qq = BY_QID[qid]
        got = list(fn(qq["text"], first_stage(qq["text"], 10)))
        if got[0] != qq["gold"]:
            wrong.append((qid, got[0], qq["gold"]))
    if wrong:
        bad("on %d of the %d spot check questions the top paragraph is not the answer" % (len(wrong), len(known)))
        for qid, got, gold in wrong:
            hint("question %d: you put [%d] first, the answer is [%d]" % (qid, got, gold))
        passed = False
    else:
        ok("on three spot check questions the true answer comes out on top")
    return report("Exercise 1, monoBERT", passed)


### Meet the cross encoder

That cell created one object, `cross_encoder`, and there is only one thing you ever do with it:

```python
cross_encoder.predict( [ (question, paragraph), (question, paragraph), ... ] )
```

You hand it a **list of pairs** and you get back **one number per pair**. Nothing else. Look at
what those numbers do.


In [ ]:
q = "How many days of paid holiday does a full time employee get?"

print("the question:", q)
print("  right paragraph:", DOCS[6]["source"], "  ", DOCS[6]["text"][:52], "...")
print("  wrong paragraph:", DOCS[18]["source"], "        ", DOCS[18]["text"][:52], "...")

one = cross_encoder.predict([(q, DOCS[6]["text"])])
two = cross_encoder.predict([(q, DOCS[6]["text"]), (q, DOCS[18]["text"])])

print("\none pair  in ->", one, " one number out")
print("two pairs in ->", two, " two numbers out")
print("\nhigher means more relevant, and the numbers can be negative.")
print("they are not probabilities yet. A sigmoid turns them into probabilities:")
print("  right paragraph: %6.2f  ->  %.3f" % (two[0], 1 / (1 + math.exp(-two[0]))))
print("  wrong paragraph: %6.2f  ->  %.3f" % (two[1], 1 / (1 + math.exp(-two[1]))))
print("\nSorting by the raw score and sorting by the probability give the SAME order,")
print("so for ranking we never bother with the sigmoid.")


Your turn, one line.


In [ ]:
# 🎯 2  Ask the cross encoder how well paragraph 0 answers this question.
#    Two things to remember from the cell above:
#      predict wants a LIST of pairs, even when you only have one pair
#      it gives back a LIST of scores, so take the first one with [0]

question = "What did we promise on delivery times in the Nordic contract?"

score = ...


print("score:", score)
check_one_score(score)


### Now the reranker

You have both halves already: how to reach into `DOCS` for a paragraph's text, and how to make
the cross encoder score a list of pairs. A reranker is those two things plus a sort.

**Two gaps to fill.** The sort underneath them is written for you. Read it rather than skip it:
lining a score up with the id it belongs to is where this kind of code usually goes wrong.


In [ ]:
def rerank_monobert(query, doc_ids):
    """Rerank candidate paragraphs by reading each one together with the question.

    query    the question, a string
    doc_ids  the candidate paragraph ids from stage one, for example [5, 0, 2, 1]
    returns  the same ids, reordered, most relevant first
    """

    # 🎯 3  Build the list of pairs the cross encoder reads.
    #    One pair per candidate id, in the same order as doc_ids, each pair being
    #    (the question, the text of that paragraph).
    #    This is exactly warm up 🎯 1 c), with (query, ...) instead of just the text.
    pairs = ...

    # 🎯 4  Let the cross encoder read every pair and score it.
    #    One call, on the whole list. You get back one number per pair.
    scores = ...

    # GIVEN, nothing to fill in here. Read it, it is the "sort" half of the method.
    #
    #   `scores[k]` is the score of `doc_ids[k]`, because we built the pairs in the
    #   same order. First we write that down as a lookup table, id -> its score:
    score_of = {}
    for position in range(len(doc_ids)):
        score_of[doc_ids[position]] = scores[position]
    #
    #   then we sort the ids by that score. `key=score_of.get` tells sorted() to look
    #   each id up in the table and sort on what it finds, and reverse=True puts the
    #   largest score first instead of the smallest.
    ranked = sorted(doc_ids, key=score_of.get, reverse=True)

    return ranked


Run the checker. It tells you what is wrong, not just that something is.


In [ ]:
check_monobert(rerank_monobert)


Now look at what changed for our question. Same ten candidates, different order.


In [ ]:
cands = first_stage(ANCHOR, 10)
show_ranking(ANCHOR, cands, k=5, title="Before, stage one alone:")
show_ranking(ANCHOR, rerank_monobert(ANCHOR, cands), k=5, title="After, reranked by monoBERT:")


And over all twelve questions.


In [ ]:
mono = evaluate(rerank_monobert, "monoBERT reranking the top 10:")
print("stage one alone:   accuracy at 1 %d/12,  MRR %.3f" % (base["acc"], base["mrr"]))
print("with monoBERT:     accuracy at 1 %d/12,  MRR %.3f" % (mono["acc"], mono["mrr"]))


### What it costs

The reranker ran once per candidate. Time it, and see what that means at the scale of a real
archive.


In [ ]:
t0 = time.time()
for _ in range(3):
    rerank_monobert(ANCHOR, first_stage(ANCHOR, 10))
per_query = (time.time() - t0) / 3
per_doc = per_query / 10

print("reranking 10 candidates:      %6.0f ms" % (per_query * 1000))
print("one paragraph:                %6.0f ms" % (per_doc * 1000))
print()
print("the whole archive (28):       %6.1f s" % (per_doc * len(DOCS)))
print("a real archive (1 million):   %6.1f hours   per question" % (per_doc * 1e6 / 3600))


That last line is the entire architecture of a modern search system, in one number.
You cannot run this model on the archive. You can run it on ten paragraphs.
So a cheap stage finds a hundred candidates, and the expensive stage judges only those.

**What you built:** the most accurate of the three, and the most expensive per candidate.
It judges one paragraph at a time, and its score is a confidence in "yes, this is relevant",
which a sigmoid turns into a probability between 0 and 1.


---
# Exercise 2. ColBERT: meet in the middle

monoBERT is accurate and slow, and it is slow for one reason: **nothing can be prepared in
advance.** The model only starts working once it can see the question and the paragraph glued
together.

ColBERT (Khattab and Zaharia, 2020) asks a cheaper question. What if we do all the reading of
the archive **last night**, and leave only a tiny comparison for question time?

The trick is what gets stored. The fast search stored **one** vector per paragraph, which is too
little: a whole page crushed into one point. monoBERT stores nothing. ColBERT stores **one vector
per word**. The paragraph is read once, last night, word by word, and every word keeps its own
small vector. At question time the only work left is comparing two piles of small vectors.

That comparison is the one formula in this notebook, and it is what you will implement.


In [ ]:
# @title Load the token encoder, the projection, and the checkers (run me, no need to read) { display-mode: "form" }
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

print("loading the token encoder (once) ...")
_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
_bert = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
for _p in _bert.parameters():
    _p.requires_grad = False           # the big model stays frozen, we only train the squeeze
_bert.eval()

DIM = 8                                # how many numbers we keep per token
torch.manual_seed(0)
projection = torch.nn.Linear(384, DIM, bias=False)   # the only thing we will train

_CACHE = {}


def bert_tokens(texts):
    """Provided. Cut each text into tokens and give every token a vector of 384 numbers.
    Padding is already stripped, so you get one clean (n_tokens, 384) block per text."""
    out = []
    for t in texts:
        if t not in _CACHE:
            b = _tok([t], padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.no_grad():
                _CACHE[t] = _bert(**b).last_hidden_state[0][b["attention_mask"][0].bool()]
        out.append(_CACHE[t])
    return out


def token_list(text):
    return _tok.convert_ids_to_tokens(_tok(text, truncation=True, max_length=128)["input_ids"])


def show_tokens(text):
    toks = token_list(text)
    vecs = bert_tokens([text])[0]
    print('"%s"' % text)
    print("  %d tokens: %s" % (len(toks), " ".join(toks)))
    print("  the encoder gives one vector per token, shape %s" % (tuple(vecs.shape),))
    print("  after your projection each of those becomes %d numbers" % DIM)


def colbert_rank(query, doc_ids):
    """Plumbing: score every candidate with YOUR maxsim and sort. Uses your encode()."""
    with torch.no_grad():
        qv = encode([query])[0]
        dvs = encode([DOCS[i]["text"] for i in doc_ids])
        scores = [float(maxsim(qv, d)) for d in dvs]
    return [i for i, s in sorted(zip(doc_ids, scores), key=lambda p: -p[1])]


def margin_on(qids):
    """How far the true answer scores above the best wrong paragraph, on average."""
    tot = 0.0
    with torch.no_grad():
        for qid in qids:
            q = BY_QID[qid]
            qv = encode([q["text"]])[0]
            cand = first_stage(q["text"], 10)
            good = float(maxsim(qv, encode([DOCS[q["gold"]]["text"]])[0]))
            others = [float(maxsim(qv, encode([DOCS[i]["text"]])[0])) for i in cand if i != q["gold"]]
            tot += good - max(others)
    return tot / len(qids)


def show_grid(query, passage):
    """Draw the grid of every question token against every passage token."""
    with torch.no_grad():
        qv, dv = encode([query, passage])
        sim = (qv @ dv.T).numpy()
    qt, dt = token_list(query), token_list(passage)
    fig, ax = plt.subplots(figsize=(0.42 * len(dt) + 2, 0.42 * len(qt) + 1.6))
    ax.imshow(sim, cmap="Blues", vmin=-1, vmax=1)
    ax.set_xticks(range(len(dt))); ax.set_xticklabels(dt, rotation=90, fontsize=8)
    ax.set_yticks(range(len(qt))); ax.set_yticklabels(qt, fontsize=8)
    for i in range(sim.shape[0]):
        j = int(sim[i].argmax())
        ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, edgecolor="crimson", lw=2))
    ax.set_xlabel("the paragraph, token by token")
    ax.set_ylabel("the question")
    ax.set_title("every question token against every paragraph token\n"
                 "red box = the best match for that row.  Their sum is the score: %.3f"
                 % float(sim.max(axis=1).sum()), fontsize=10)
    plt.tight_layout(); plt.show()


def maxsim_fast(q_vectors, d_vectors):
    """The same formula as yours, written as one matrix product. This is what a real
    system runs. Used here only to show that the two agree."""
    return (q_vectors @ d_vectors.T).max(dim=1).values.sum()


def check_maxsim(fn):
    print("checking your maxsim ...\n")
    q = torch.tensor([[1., 0.], [0., 1.], [0.6, 0.8]])   # 3 question tokens
    d = torch.tensor([[1., 0.], [0., 1.]])               # 2 paragraph tokens
    #  grid = [[1.0, 0.0],
    #          [0.0, 1.0],
    #          [0.6, 0.8]]
    #  best in each row = 1.0, 1.0, 0.8   and   1.0 + 1.0 + 0.8 = 2.8
    try:
        got = float(fn(q, d))
    except Exception as e:
        bad("your function raised %s: %s" % (type(e).__name__, e))
        hint("a 🎯 gap is probably still a bare `...`")
        return report("maxsim", False)
    diagnoses = [
        (2.8, None),
        (0.6, "you kept the WORST match in each row instead of the best. Turn the comparison "
              "in b) around."),
        (0.8, "you replaced the total each time instead of adding to it, so you kept only the "
              "last question token. In c) start from `total` and add to it."),
        (3.4, "you added every cell of the grid. The adding belongs in the OUTER loop, after "
              "the best match for that question token has been found."),
        (2.0, "you kept the best QUESTION token for each paragraph token. The formula runs the "
              "other way round: for each question token, the best paragraph token."),
        (1.0, "you kept only the single largest cell of the whole grid, not one winner per row."),
    ]
    for value, msg in diagnoses:
        if abs(got - value) < 1e-3:
            if msg is None:
                ok("the hand computed example gives 2.8, and so does yours")
                break
            bad("your answer is %.3f, the right answer is 2.800" % got)
            hint(msg)
            return report("maxsim", False)
    else:
        bad("your answer is %.4f, the right answer is 2.800" % got)
        hint("grid = [[1, 0], [0, 1], [0.6, 0.8]]. Best in each row: 1, 1, 0.8. Sum: 2.8")
        return report("maxsim", False)

    if float(fn(q, d)) == float(fn(d, q)):
        bad("your score is the same both ways round, it should not be")
        hint("swapping the question and the paragraph changes how many winners you add up")
        return report("maxsim", False)
    ok("asymmetric, as it should be: one winner per question token, not per paragraph token")

    try:
        s = fn(encode([ANCHOR])[0], encode([DOCS[0]["text"]])[0])
    except Exception as e:
        bad("your maxsim works on the hand example but not on real text: %s" % e)
        return report("maxsim", False)
    if torch.is_tensor(s) and s.dim() != 0:
        bad("on real vectors your score is a %s, it has to be one single number" % list(s.shape))
        hint("after taking the best match in each row you still have one number per question "
             "token. Add them up.")
        return report("maxsim", False)
    if not torch.is_tensor(s):
        bad("your score is a plain %s, so PyTorch cannot train through it" % type(s).__name__)
        hint("do not call float() or .item() inside maxsim, keep the torch numbers as they are")
        return report("maxsim", False)
    if abs(float(s) - float(maxsim_fast(encode([ANCHOR])[0], encode([DOCS[0]["text"]])[0]))) > 1e-3:
        bad("on real text your score is %.3f, the formula gives %.3f"
            % (float(s), float(maxsim_fast(encode([ANCHOR])[0], encode([DOCS[0]["text"]])[0]))))
        return report("maxsim", False)
    ok("on real text the score is a single number: %.3f" % float(s))
    return report("Exercise 2, the best match score", True)


def check_training(before, after, history):
    print("checking the training ...\n")
    passed = True
    if len(history) < 2:
        bad("the training loop did not finish, there is no loss curve to look at")
        hint("scroll up: the cell above should have printed one line per epoch")
        return report("Exercise 2, the training loop", False)
    if history[-1] >= history[0]:
        bad("the loss did not go down (%.4f then %.4f)" % (history[0], history[-1]))
        hint("the three lines of 🎯 8 are what actually change the parameters. Without "
             "loss.backward() and optimizer.step() nothing moves.")
        passed = False
    elif history[-1] > history[0] / 5:
        bad("the loss went down only a little (%.4f to %.4f)" % (history[0], history[-1]))
        hint("check the loss in 🎯 7: it should be small when score_right is the LARGER one")
        passed = False
    else:
        ok("the loss fell from %.4f to %.4f" % (history[0], history[-1]))

    if after["margin"] <= before["margin"]:
        bad("the true answer did not pull further ahead (%+.2f then %+.2f)"
            % (before["margin"], after["margin"]))
        hint("if the loss fell but the margin did not grow, check that score_right and "
             "score_wrong in 🎯 6 are not the wrong way round")
        passed = False
    else:
        ok("the true answer pulled ahead of the best wrong paragraph: %+.2f to %+.2f"
           % (before["margin"], after["margin"]))

    if after["acc"] < before["acc"]:
        bad("accuracy on the logged questions went down, %d to %d" % (before["acc"], after["acc"]))
        passed = False
    else:
        ok("accuracy on the logged questions: %d/%d to %d/%d"
           % (before["acc"], before["n"], after["acc"], after["n"]))
    return report("Exercise 2, the training loop", passed)


### Step 1: from words to vectors

A transformer cuts text into **tokens** (roughly words, sometimes pieces of words) and gives
each one a vector of 384 numbers. Look at it for our question.


In [ ]:
show_tokens(ANCHOR)
print()
show_tokens("Nordic Supply Agreement: deliver within 5 working days.")


384 numbers per token is far too much to store for a whole archive. A million paragraphs at
60 tokens each would be 90 gigabytes. So ColBERT adds one small layer that **squeezes** each
token vector down to a handful of numbers. We use 8.

That squeeze is the only part we will train. The big transformer stays frozen.

Here is the whole of it, written out for you. Two lines: apply the squeeze, then rescale every
vector to length 1. The second line matters more than it looks, because once two vectors both
have length 1, their dot product **is** their cosine similarity, and that is the "how close are
these two words" number the formula needs.


In [ ]:
def encode(texts):
    """GIVEN. Turn each text into one small vector per token.

    texts    a list of strings
    returns  a list with one block per text, each of shape (number of tokens, DIM)
    """
    out = []
    for token_vectors in bert_tokens(texts):        # (number of tokens, 384)
        small = projection(token_vectors)           # squeeze 384 numbers down to DIM
        small = F.normalize(small, dim=-1)          # rescale every row to length 1
        out.append(small)
    return out


# look at what it produces
question_vectors = encode([ANCHOR])[0]
print("the question has %d tokens, and each one is now a vector of %d numbers"
      % (question_vectors.shape[0], question_vectors.shape[1]))
print("the vector of the first token:", [round(float(x), 3) for x in question_vectors[0]])
print("its length:", round(float(question_vectors[0].norm()), 3), "(every row has length 1)")


### Step 2: the best match score

Now the formula, and this one you write. In words:

> **For every word of the question, find its best matching word in the paragraph.
> Then add those best matches up.**

That is all it is. Written out:

$$S(q,d) \;=\; \sum_{i \in \text{question}} \; \max_{j \in \text{paragraph}} \; q_i \cdot d_j$$

Read it as two loops and a running total:

1. for every **question** token, look at every **paragraph** token and work out how similar they
   are. That is the **grid**: one row per question word, one column per paragraph word;
2. in each row keep only the **largest** number, the best match for that question word;
3. **add** those winners together.

The direction matters. One winner per **question** word, not per paragraph word. A long paragraph
does not get a bigger score just for being long, but a long question does, and that is a real
property of this score which we come back to at the end.

The skeleton below is the two loops. You fill in three things: the similarity of one pair of
words, the comparison that keeps the best one, and the adding up.


In [ ]:
def maxsim(q_vectors, d_vectors):
    """The best match score.

    q_vectors   the question, one row per question token
    d_vectors   the paragraph, one row per paragraph token
    returns     one single number
    """
    total = 0.0

    # one round of this loop per question token
    for i in range(len(q_vectors)):

        best_for_this_token = None

        # look at every paragraph token and remember the best match
        for j in range(len(d_vectors)):

            # 🎯 5 a) How similar are question token i and paragraph token j?
            #    Both are vectors of DIM numbers and both have length 1, so their dot
            #    product is exactly their cosine similarity.  torch.dot(a, b) computes it.
            similarity = ...

            # 🎯 5 b) Keep this one if it beats the best match found so far for question
            #    token i. The first one always wins, because best_for_this_token is None.
            #    Fill in the comparison after the `or`.
            if best_for_this_token is None or ...:
                best_for_this_token = similarity

        # 🎯 5 c) This question token is done. Add its best match to the running total.
        total = ...

    return total


In [ ]:
check_maxsim(maxsim)


Your version walks the grid one cell at a time, which is the clearest way to read the formula and
the slowest way to run it. A real system computes the same grid as a single matrix product. Same
number, and it is worth seeing how much that costs.


In [ ]:
qv, dv = encode([ANCHOR, DOCS[0]["text"]])

t0 = time.time()
for _ in range(20):
    mine = maxsim(qv, dv)
loop_ms = (time.time() - t0) / 20 * 1000

t0 = time.time()
for _ in range(20):
    theirs = maxsim_fast(qv, dv)
fast_ms = (time.time() - t0) / 20 * 1000

print("your two loops:      %.3f   %6.2f ms" % (float(mine), loop_ms))
print("one matrix product:  %.3f   %6.2f ms" % (float(theirs), fast_ms))
print("\nsame number, %.0f times faster." % (loop_ms / fast_ms))
print("The formula is what you wrote. The matrix product is only how it is spelled in a")
print("library. We keep YOUR version for the training below, which is why it takes a minute.")


Here is the grid your formula just reduced. Every cell is one question word against one paragraph
word. The red boxes are the row winners, and their sum is the score.


In [ ]:
show_grid("delivery times Nordic contract",
          "Nordic Supply Agreement: deliver within 5 working days.")


### Step 3: teach the squeeze

The squeeze started as a random layer. A random way of throwing away 376 of 384 numbers keeps
some of the meaning by luck, but nothing more. We are going to train it: keep the eight numbers
that make right paragraphs score above wrong ones.

The training data is the cheapest kind a company has, a **query log**: the questions people
actually typed, and for each one, which paragraph turned out to answer it.

From that we build **triples**. A triple is three pieces of text:

> `(the question, a paragraph that answers it, a paragraph that does not)`

and it carries exactly one instruction to the model: **the second one should score higher than
the third one.** Nothing more. We never tell it what the score should be, only which of the two
should win. That is enough, because ranking only ever depends on which score is bigger.

Three details about how we build them, each of which is what a real team would also do:

* **the wrong paragraph is not random.** It is one of the paragraphs the fast search itself put
  near the top for that question. A randomly chosen paragraph is trivially easy to reject and
  teaches nothing. These are the near misses you browsed at the start;
* **one question gives many triples.** Eight logged questions, each with nine different wrong
  paragraphs, and each asked in four different phrasings, gives 288 triples;
* **`TRIPLES` is just a Python list.** Each entry is a tuple of three strings. No special format,
  no library.


In [ ]:
# @title Build the training triples (run me, no need to read) { display-mode: "form" }
TRIPLES = []
_pairs = []
for qid in TRAIN_QIDS:
    _pairs.append((BY_QID[qid]["text"], qid))
    for p in PARAPHRASES[qid]:
        _pairs.append((p, qid))
for text, qid in _pairs:
    gold = BY_QID[qid]["gold"]
    for wrong in [i for i in first_stage(BY_QID[qid]["text"], 10) if i != gold][:9]:
        TRIPLES.append((text, DOCS[gold]["text"], DOCS[wrong]["text"]))

print("%d questions in the log (the 8 logged ones, each asked in 4 different ways)" % len(_pairs))
print("%d training triples" % len(TRIPLES))
print("%d parameters to train (the squeeze is a 384 by %d matrix)\n" % (384 * DIM, DIM))
print("one triple looks like this:")
_q, _right, _wrong = TRIPLES[0]
print('  question:  "%s"' % _q)
print('  right:     "%s..."' % _right[:70])
print('  wrong:     "%s..."' % _wrong[:70])


Before training, measure. Your `encode` and your `maxsim` are what is being measured here.


In [ ]:
before = evaluate(colbert_rank, "ColBERT before training, on the 8 questions in the log:",
                  qids=TRAIN_QIDS)
before["margin"] = margin_on(TRAIN_QIDS)
print("the true answer scores %+.3f above the best wrong paragraph, on average" % before["margin"])


### The loss: turning "this one should win" into a number

Training needs a single number that says how wrong the model currently is, and that gets *smaller*
as the model gets *better*. We have two scores per triple, $s_{\text{right}}$ and
$s_{\text{wrong}}$. Here is how they become one number.

First turn the two scores into a **share**: how much of the total belongs to the right paragraph.

$$p \;=\; \frac{e^{\,s_{\text{right}}}}{e^{\,s_{\text{right}}} + e^{\,s_{\text{wrong}}}}$$

The $e^{(\cdot)}$ is there to make both quantities positive, so the fraction behaves like a share
of a whole, and it stays well defined even when a score is negative. $p$ is always strictly
between 0 and 1, it is near 1 when the right paragraph is well ahead, and it is exactly 0.5 when
the two scores are equal.

Then the loss is

$$\text{loss} \;=\; -\log p$$

The logarithm turns "a share near 1" into "a loss near 0", and it punishes confident mistakes very
hard:

| $s_{\text{right}}$ | $s_{\text{wrong}}$ | share $p$ | loss | what it means |
|---|---|---|---|---|
| 8 | 2 | 0.998 | 0.002 | right paragraph well ahead, almost nothing to fix |
| 5 | 5 | 0.500 | 0.693 | the model is guessing |
| 2 | 8 | 0.002 | 6.002 | confidently wrong, and the loss is enormous |

Two things worth noticing, because they explain the rest of this notebook.

**Only the difference matters.** Scores of 8 and 2 give exactly the same loss as scores of 108 and
102. Nothing in this loss asks the score to be large, or to sit between 0 and 1, or to mean
anything on its own. It only ever asks the right paragraph to be **ahead**. That is why ColBERT's
score is not a probability and why a threshold like "keep everything above 0.7" is meaningless
for it.

**It never fully stops.** The loss can get very small, but it is never zero, so training keeps
pushing the gap wider for as long as you let it run. Watch it do exactly that in a moment.

This is the pairwise softmax cross entropy the ColBERT paper uses. You are about to write it in
one line.


In [ ]:
# the same three rows as the table above, computed rather than asserted
print("  s_right  s_wrong    share p     loss")
for s_right, s_wrong in [(8.0, 2.0), (5.0, 5.0), (2.0, 8.0)]:
    r, w = torch.tensor(s_right), torch.tensor(s_wrong)
    p = torch.exp(r) / (torch.exp(r) + torch.exp(w))
    print("  %7.1f  %7.1f   %8.3f  %7.3f" % (s_right, s_wrong, p, -torch.log(p)))


### The training loop

Three gaps: score both paragraphs, turn the two scores into the loss above, and then the three
lines of PyTorch that do the actual learning.

Those three lines are the same in every PyTorch program ever written, so they are worth
memorising. Each triple goes through the same cycle:

1. **clear** the gradients. PyTorch *adds* new gradients to whatever is already stored, so if you
   forget this step, each triple gets trained on together with every triple before it;
2. **work out the gradients** for the current loss, that is, for each parameter, which direction
   would make this loss smaller. This is the step that runs backwards through everything you
   computed, which is why it is called backward;
3. **take the step**: the optimizer nudges every parameter a little way in that direction.

The three method calls you need, in some order, are `loss.backward()`, `optimizer.step()` and
`optimizer.zero_grad()`. Match each one to the description in the code.

It runs your `maxsim` about three and a half thousand times, so give it a minute or two.


In [ ]:
EPOCHS = 6
optimizer = torch.optim.Adam(projection.parameters(), lr=0.005)
loss_history = []
random.seed(0)

for epoch in range(EPOCHS):
    random.shuffle(TRIPLES)
    running = 0.0

    for question, right_paragraph, wrong_paragraph in TRIPLES:
        q_vec, right_vec, wrong_vec = encode([question, right_paragraph, wrong_paragraph])

        # 🎯 6  Score both paragraphs against the question, with your own formula.
        score_right = ...
        score_wrong = ...

        # 🎯 7  The loss, exactly the formula above. Two lines:
        #     the share that belongs to the RIGHT paragraph, then minus its logarithm.
        #     Use torch.exp(...) and torch.log(...).
        share = ...
        loss = ...

        # 🎯 8  The three lines that actually make PyTorch learn.
        #     Use each of these ONCE, and work out which goes where:
        #
        #          loss.backward()      optimizer.step()      optimizer.zero_grad()
        #
        #  a) clear the gradients left over from the previous triple, because PyTorch
        #     adds new ones to whatever is already there
        ...

        #  b) work out, for every parameter, which direction makes THIS loss smaller
        ...

        #  c) nudge every parameter one small step in that direction
        ...

        running += float(loss)

    loss_history.append(running / len(TRIPLES))
    print("epoch %d/%d   loss %.4f" % (epoch + 1, EPOCHS, loss_history[-1]))


In [ ]:
after = evaluate(colbert_rank, "ColBERT after training, on the 8 questions in the log:",
                 qids=TRAIN_QIDS)
after["margin"] = margin_on(TRAIN_QIDS)
print("the true answer now scores %+.3f above the best wrong paragraph\n" % after["margin"])

plt.figure(figsize=(5, 3))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("the loss falling")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

check_training(before, after, loss_history)


### What that number does not mean

The loss went to almost nothing and the eight logged questions are now perfect. Before believing
it, look at the four questions that were deliberately kept out of the query log.


In [ ]:
unseen = evaluate(colbert_rank, "The four questions nobody ever logged:", qids=TEST_QIDS)
print("on the questions it trained on:  %d/%d" % (after["acc"], after["n"]))
print("on questions it has never seen:  %d/%d" % (unseen["acc"], unseen["n"]))


Worse, most likely, than before training started.

This is **overfitting**, and it is worth seeing once with your own hands, because it is the single
most common way a company wastes a month on a reranker. We fitted 3072 parameters to 288 triples
that came from **eight** distinct questions. The model did not learn what relevance is. It learned
those eight questions.

The real ColBERT is trained on millions of triples covering tens of thousands of distinct
questions. The mechanism you just wrote is exactly the right one. The data is a toy.

The practical rule: if you are about to fine tune a reranker on the fifty questions your team
happened to write down, do not then measure it on those fifty questions.

**What you built:** a score that can be prepared in advance, so it is far cheaper than monoBERT at
question time. Note what it returns: a **sum of cosines**, not a probability. It has no ceiling of
1, and it grows with the length of the question. Remember that for the end.


---
# Exercise 3. RankGPT: ask for the order, not for a score

The first two rerankers both produced a **number** for each paragraph, one paragraph at a time,
and then we sorted by that number. Each paragraph was judged alone, in the dark, never against
its rivals.

RankGPT (Sun and others, 2023) does something else. It hands a language model the whole shortlist
at once, numbered, and asks for the **order**:

```
[1] Warehouse Handbook, chapter 4. Typical shipment lead time ...
[2] Nordic Supply Agreement, section 7. Alpine Instruments shall deliver ...
[3] Nordic Supply Agreement, section 9. The customer shall settle every invoice ...

Rank the passages by relevance to: what did we promise on delivery times
in the Nordic contract?
```

and the model answers

```
[2] > [1] > [3]
```

No score comes back. Just an order. And that is the interesting part: the paragraphs are now
compared **against each other**, in one reading. The model can notice that paragraph 2 is the
promise and paragraph 3 is the same contract talking about something else. A pointwise scorer
never gets to make that comparison.

**The catch:** the shortlist is 100 paragraphs and they do not fit into one prompt, or if they
do fit, quality falls apart in a long list. So RankGPT sorts a **window** at a time, and slides
it. That sliding is what you implement.


In [ ]:
# @title The prompt, the reply parser, and the checker (run me, no need to read) { display-mode: "form" }
PROMPT = """I will provide you with {n} passages, each with a number in square brackets.
Rank them by how well they answer the search query.

Search query: {query}

{passages}

Rank the {n} passages by relevance to the search query, the most relevant first.
Answer only with the numbers in the format [4] > [1] > [2], and nothing else."""


def build_prompt(query, texts):
    passages = "\n".join("[%d] %s" % (i + 1, t) for i, t in enumerate(texts))
    return PROMPT.format(n=len(texts), query=query, passages=passages)


def parse_permutation(reply, n):
    """Turn '[3] > [1] > [2]' into [2, 0, 1]. Repairs whatever the model got wrong.

    Language models drop identifiers, repeat them, and invent new ones. The paper does not
    say what to do about it, so every implementation makes its own repair. Ours: keep the
    first mention of each valid number, then append anything the model forgot, in its old
    order."""
    seen, order = set(), []
    for m in re.findall(r"\[(\d+)\]", reply):
        k = int(m) - 1
        if 0 <= k < n and k not in seen:
            seen.add(k)
            order.append(k)
    order += [k for k in range(n) if k not in seen]
    return order


LAST_EXCHANGE = {}


def llm_rank_window(query, doc_ids):
    """Ask the language model to put these few paragraphs in order."""
    prompt = build_prompt(query, [DOCS[i]["text"] for i in doc_ids])
    reply = call_gemini(prompt)
    order = parse_permutation(reply, len(doc_ids))
    LAST_EXCHANGE.update(prompt=prompt, reply=reply, ids_in=list(doc_ids),
                         ids_out=[doc_ids[k] for k in order])
    return [doc_ids[k] for k in order]


def local_rank_window(query, doc_ids):
    """Offline stand in for the language model, so the notebook works without a key.
    It sorts the window with the cross encoder from exercise 1 and then writes out the
    same kind of exchange a real model would have produced."""
    doc_ids = list(doc_ids)
    ids_out = rerank_monobert(query, doc_ids)
    reply = " > ".join("[%d]" % (doc_ids.index(i) + 1) for i in ids_out)
    LAST_EXCHANGE.update(prompt=build_prompt(query, [DOCS[i]["text"] for i in doc_ids]),
                         reply=reply, ids_in=doc_ids, ids_out=ids_out)
    return ids_out


def backend_name():
    return ("Gemini, %s" % MODEL) if USE_REAL_LLM else \
           "the offline stand in (the cross encoder from exercise 1)"


def rank_window(query, doc_ids):
    """Sort ONE window. This is the thing your sliding window will call."""
    return llm_rank_window(query, doc_ids) if USE_REAL_LLM else local_rank_window(query, doc_ids)


def show_exchange():
    if not LAST_EXCHANGE:
        print("no window has been sorted yet, run the cell above first")
        return
    print("who sorted it: %s\n" % backend_name())
    print("THE PROMPT (shortened)\n")
    lines = LAST_EXCHANGE["prompt"].split("\n")
    for ln in lines[:6]:
        print("  " + ln[:100])
    print("  ...")
    for ln in lines[-3:]:
        print("  " + ln[:100])
    print("\nWHAT THE MODEL ANSWERED\n")
    print("  " + LAST_EXCHANGE["reply"][:300])
    print("\n  read as: %s" % LAST_EXCHANGE["ids_out"])


def show_window_plan(n=20, window=8, step=4):
    """Draw which positions each pass looks at. Every position is 3 columns wide, so the
    bars line up with the numbers above them by construction."""
    label_width = 9
    print(" " * 0 + "positions".ljust(label_width) + "".join("%3d" % i for i in range(n)))
    start, p = n - window, 1
    while start >= 0:
        inside = ("sort %d to %d" % (start, start + window - 1)).center(3 * window - 2, "=")
        print(("pass %d" % p).ljust(label_width + 3 * start) + "[" + inside + "]")
        start -= step
        p += 1
    print("\nthe window moves %d positions each pass, so consecutive windows overlap by %d."
          % (step, window - step))


def check_sliding_window(fn):
    print("checking your sliding window ...\n")
    ids = ["p%d" % i for i in range(20)]
    truth = {d: 1.0 - 0.01 * i for i, d in enumerate(ids)}
    truth["p19"] = 10.0        # the answer, parked at the very back of the list
    truth["p18"] = 9.0
    calls = []

    def perfect_ranker(query, chunk):
        calls.append(list(chunk))
        return sorted(chunk, key=lambda d: -truth[d])

    try:
        out = fn("any question", ids, perfect_ranker, window=8, step=4)
    except TypeError as e:
        bad("your function raised TypeError: %s" % e)
        hint("if it mentions 'ellipsis', one of the 🎯 gaps is still a bare `...`")
        return report("the sliding window", False)
    except Exception as e:
        bad("your function raised %s: %s" % (type(e).__name__, e))
        return report("the sliding window", False)

    out = list(out) if out is not None else []
    if sorted(out) != sorted(ids):
        bad("the result must be the same 20 paragraphs, only reordered. Got %d of them." % len(out))
        hint("put the reordered window back into `ranking` at the same positions")
        return report("the sliding window", False)
    ok("the whole list comes back, reordered")

    if len(calls) == 0:
        bad("the model was never called")
        return report("the sliding window", False)
    if len(calls) == 1 and len(calls[0]) == 20:
        bad("you asked the model to sort all 20 paragraphs in one go")
        hint("that is the thing the sliding window exists to avoid. Sort 8 at a time.")
        return report("the sliding window", False)
    if len(calls) != 4:
        bad("the model was called %d times, it should be 4" % len(calls))
        hint("20 paragraphs, a window of 8, sliding 4 at a time: 12 to 19, 8 to 15, 4 to 11, 0 to 7")
        return report("the sliding window", False)
    ok("4 calls, one per window, each looking at 8 paragraphs")

    if calls[0][0] != "p12":
        bad("the first window started at %s, it should start at the BACK, at p12" % calls[0][0])
        hint("the first window covers the last 8 positions, so it starts at len(list) - window")
        return report("the sliding window", False)
    ok("the first window sits at the back of the list")

    where = out.index("p19")
    if where != 0:
        bad("the best paragraph ended at position %d instead of 0" % where)
        if where == 12:
            hint("your window is sliding from the front to the back. Going that way, the best "
                 "paragraph can only move up by one window. Start at the back and slide forward.")
        else:
            hint("windows must overlap, so a winner is carried into the next window and can keep "
                 "climbing. With window 8 and step 4 they overlap by 4.")
        return report("the sliding window", False)
    ok("the best paragraph travelled from position 19 all the way to position 0")
    return report("Exercise 3, the sliding window", True)


### The sliding window

Twenty candidates, a model that reads eight at a time, moving four positions each pass. Run the
cell to see the plan.


In [ ]:
show_window_plan(n=20, window=8, step=4)


**The direction is the whole trick.** Start at the back and walk forwards. Because the windows
overlap by four, a paragraph that wins its window is carried into the next window, which sits
closer to the front, where it can win again. A genuinely good paragraph sitting at position 19
can travel all the way to position 0 in four passes.

Follow one: it starts at position 19, pass 1 sorts it to position 12, pass 2 covers 8 to 15 so it
moves to 8, pass 3 covers 4 to 11 so it moves to 4, and pass 4 brings it to 0.

Go the other way, front to back, and that same paragraph can never move more than one window's
worth. The checker will tell you if you get this backwards.

Try changing the numbers in the cell above (a longer list, a bigger window, a smaller step) and
watch how many passes it takes.


In [ ]:
def sliding_window_rerank(query, doc_ids, rank_window, window=8, step=4):
    """Rerank a long list with a model that can only look at `window` paragraphs at a time.

    query        the question
    doc_ids      the whole candidate list, for example 20 ids
    rank_window  a function rank_window(query, some_ids) that returns those ids reordered
    window       how many paragraphs the model sees at once
    step         how far the window moves each time
    """
    ranking = list(doc_ids)

    # 🎯 9  Where does the FIRST window start?
    #    It sits at the very back of the list, covering the last `window` positions.
    #    With 20 paragraphs and a window of 8, it should start at position 12.
    start = ...

    while start >= 0:
        # 🎯 10  Three lines, the body of one window.
        #  a) Cut out the paragraphs this window covers: the slice from `start` up to
        #     `start + window`. In Python that is ranking[a:b].
        chunk = ...

        #  b) Hand that chunk to the model and take back the reordered version.
        chunk = ...

        #  c) Put it back into `ranking`, in the new order, in the same positions.
        #     You can assign to a slice: ranking[a:b] = something
        ...

        # 🎯 11  Slide the window towards the FRONT of the list, by `step` positions.
        start = ...

    return ranking


The checker runs your loop against a **perfect** sorter instead of a real model: no key needed,
no cost, and completely repeatable. It parks the best paragraph at position 19 and sees whether
your window brings it home.


In [ ]:
check_sliding_window(sliding_window_rerank)


### Now with a real language model

Get a free key, it takes two minutes:

1. go to https://aistudio.google.com/apikey and create an API key;
2. in Colab, click the **key icon** in the left sidebar (Secrets);
3. add a secret named `GEMINI_API_KEY`, paste the key, and switch on **Notebook access**.

If you skip this, everything below still runs: an offline stand in plays the part of the language
model, so nobody is stuck.

One thing to know before you start spending it. The free tier allows **five calls a minute** for
this model, and a sliding window is one call per window, so this exercise runs into that limit
almost immediately. The cell below therefore paces itself: it counts its own calls and waits when
it is at the limit, rather than letting the API refuse. That is a good habit in any case, and it
is also the first honest thing this method teaches you. The other two rerankers had no such bill.


In [ ]:
import os

MODEL = "gemini-2.5-flash"
CALLS_PER_MINUTE = 5      # what the free tier allows for this model. Raise it on a paid key.

USE_REAL_LLM = False
client = None

_key = None
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
except Exception:
    _key = os.environ.get("GEMINI_API_KEY")       # so the same code runs outside Colab

if _key:
    from google import genai
    client = genai.Client(api_key=_key)
    USE_REAL_LLM = True
    print("connected to %s, free tier budget %d calls a minute" % (MODEL, CALLS_PER_MINUTE))
else:
    print("no GEMINI_API_KEY found")
    print("falling back to the offline stand in, everything below still works")

try:
    from google.genai import types
    _CFG = types.GenerateContentConfig(temperature=0.0,
                                       thinking_config=types.ThinkingConfig(thinking_budget=0))
except Exception:
    _CFG = None

_call_times = []


def _wait_for_quota():
    """Never make more than CALLS_PER_MINUTE calls in any 60 seconds, so the free tier
    never has to refuse us. Sleeps, out loud, when we are at the limit."""
    while True:
        now = time.time()
        while _call_times and now - _call_times[0] > 60:
            _call_times.pop(0)
        if len(_call_times) < CALLS_PER_MINUTE:
            _call_times.append(now)
            return
        wait = 60 - (now - _call_times[0]) + 1
        print("   (free tier: %d calls a minute. Waiting %.0f s.)" % (CALLS_PER_MINUTE, wait))
        time.sleep(wait)


def _retry_seconds(message, fallback):
    """The server usually tells us exactly how long to wait. Believe it."""
    for pattern in (r"retryDelay['\"]?:\s*['\"]?(\d+(?:\.\d+)?)s", r"retry in (\d+(?:\.\d+)?)s"):
        m = re.search(pattern, message)
        if m:
            return float(m.group(1)) + 1
    return fallback


def call_gemini(prompt, retries=5):
    """One call, staying inside the free tier and backing off when told to."""
    delay, cfg = 10.0, _CFG
    for attempt in range(retries):
        _wait_for_quota()
        try:
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            return (r.text or "").strip()
        except Exception as e:
            s = str(e)
            if cfg is not None and ("thinking" in s.lower() or "INVALID_ARGUMENT" in s):
                cfg = None            # this model does not take the thinking setting
                continue
            transient = any(t in s for t in ("429", "RESOURCE_EXHAUSTED", "503",
                                             "UNAVAILABLE", "500", "INTERNAL"))
            if not transient or attempt == retries - 1:
                raise
            wait = _retry_seconds(s, delay)
            print("   (the API asked us to slow down. Waiting %.0f s.)" % wait)
            time.sleep(wait)
            _call_times.clear()       # that wait cleared the window
            delay = min(delay * 2, 60)
    raise RuntimeError("gave up after %d attempts" % retries)


Run your sliding window over the top 20 candidates for our question.


In [ ]:
candidates = first_stage(ANCHOR, 20)
show_ranking(ANCHOR, candidates, k=5, title="Before, straight from the fast search:")

t0 = time.time()
ordered = sliding_window_rerank(ANCHOR, candidates, rank_window, window=8, step=4)
print("4 windows in %.1f s\n" % (time.time() - t0))

show_ranking(ANCHOR, ordered, k=5, title="After RankGPT:")


Here is the prompt that was built for the last window, and the answer that came back.


In [ ]:
show_exchange()


Notice what the model returned: identifiers, nothing else. No score, no probability, no
confidence. Also notice that nothing forces it to return a valid permutation. It can repeat a
number, skip one, or invent one, and the paper does not say what to do when it does. Our
`parse_permutation` quietly repairs it. Every production implementation has its own repair rule,
which is worth knowing before you trust one.

Now over several questions, and here the bill arrives. Every window is one API call, and the free
tier allows five calls a minute. So the cell below shortens the candidate list to 12, which is two
windows per question, and on a real key it takes two questions rather than four. If it runs out of
quota it pauses and says so. Nothing is broken when that happens: that pause is the cost of this
method, made visible.


In [ ]:
def rankgpt_rank(query, doc_ids):
    return sliding_window_rerank(query, doc_ids, rank_window, window=8, step=4)

# 12 candidates with a window of 8 is 2 calls per question. On a real key we take 2 questions,
# so 4 calls in total. The offline stand in costs nothing, so it takes all 4 questions.
subset = TEST_QIDS[:2] if USE_REAL_LLM else TEST_QIDS

print("sorted by: %s" % backend_name())
print("%d questions, 2 windows each, so %d calls\n" % (len(subset), 2 * len(subset)))

rg = evaluate(rankgpt_rank, "RankGPT on questions the fast search gets wrong:",
              qids=subset, depth=12)
print("stage one alone on the same questions:  accuracy at 1 %d/%d"
      % (evaluate(None, "", qids=subset, depth=12, quiet=True)["acc"], len(subset)))


If you ran the offline stand in, be careful what you conclude: what you just measured is the cross
encoder from exercise 1 driven through a sliding window. The loop is RankGPT's. The judge is not.

**What you built:** a reranker with no training, no labelled data and no model of your own. That
is its real selling point. It is also the slowest and the only one that sends your documents to
somebody else's server, which for a contract archive is a conversation with your legal team, not
a technical detail.

One honest number from the paper, worth carrying out of the room: with the strongest model of the
day this method beat every trained reranker. With the cheap model it **lost** to a trained
reranker a fraction of its size. "Use an LLM to rank" is not a strategy on its own.


---
# Where the answer lands in the prompt

A reranker has two jobs, not one. Dropping the wrong paragraphs is the obvious one. The second is
**the order of the few that survive**, because that order becomes the order of the prompt, and a
language model does not read a prompt evenly. "Lost in the Middle" (Liu and others, 2023) measured
it: the same model, given the same documents, answered best when the useful one sat at the very
beginning, worst when it sat in the middle.

So start with the deterministic half. Where does the true answer sit in the five paragraphs we
paste into the prompt, before and after reranking?


In [ ]:
print("        stage one          after monoBERT")
print("  q     position in the prompt")
s1_first = mono_first = missing = 0
for q in QUERIES:
    cand = first_stage(q["text"], 10)
    top5_before = cand[:5]
    top5_after = rerank_monobert(q["text"], cand)[:5]
    a = top5_before.index(q["gold"]) + 1 if q["gold"] in top5_before else 0
    b = top5_after.index(q["gold"]) + 1 if q["gold"] in top5_after else 0
    s1_first += a == 1
    mono_first += b == 1
    missing += a == 0
    fmt = lambda p: ("not in the 5" if p == 0 else "%d" % p)
    print("  %2d    %-18s %s" % (q["qid"], fmt(a), fmt(b)))

print("\n  answer sitting first in the prompt:  %d/12  ->  %d/12" % (s1_first, mono_first))
print("  answer not pasted in at all:         %d/12  ->  0/12" % missing)


That is the whole value of a reranker in one table, and none of it needed a language model to
measure. Cutting from 10 to 5 either keeps the answer or throws it away, and the order of the
survivors decides how hard the model has to work to see it.

### Optional: does the position actually change the answer?

The U shaped curve from the paper is a **2023 measurement on 2023 models**, and it varies wildly
from model to model. So do not take it on faith. Measure it on the model you actually use.

The cell below takes one question, builds the same five paragraph prompt five times with the
answer at a different position each time, and asks the model. Five calls. It needs a key, and it
is honest about whatever it finds, including "no effect at this size".


In [ ]:
if not USE_REAL_LLM:
    print("no key, skipping. Everything above already ran without one.")
else:
    question, key_fact = ANCHOR, "5 working"
    cand = rerank_monobert(question, first_stage(question, 10))
    gold = gold_of(question)
    others = [i for i in cand if i != gold][:4]

    print("question: %s" % question)
    print("looking for the fact: '%s ... days'\n" % key_fact)
    for pos in range(5):
        ids = others[:pos] + [gold] + others[pos:]
        passages = "\n\n".join("[%d] %s" % (n + 1, DOCS[i]["text"]) for n, i in enumerate(ids))
        prompt = ("Answer the question using only the passages below. One short sentence.\n\n"
                  "%s\n\nQuestion: %s" % (passages, question))
        answer = call_gemini(prompt)
        found = key_fact in answer.lower()
        print("  answer at position %d:  %s  %s" % (pos + 1, "correct" if found else "WRONG   ",
                                                    answer.replace("\n", " ")[:90]))
    print("\nIf every position was correct, that is a real result too: at five short paragraphs")
    print("this model does not care where the answer sits. Run it again with 20 paragraphs, or")
    print("with a smaller model, and the picture changes. That is the point: measure your own.")


---
# What you built

| | Reranker | What it returns | Prepared in advance | Cost per question |
|---|---|---|---|---|
| 1 | **monoBERT** | a probability between 0 and 1 | nothing | one model run per candidate |
| 2 | **ColBERT** | a sum of cosines, no ceiling | one small vector per word | a matrix product |
| 3 | **RankGPT** | no number at all, only an order | nothing | a few API calls |

Read the third column again, because it is the thing that survives this notebook:

**What comes out of a reranker is not one kind of thing.** A cutoff you tuned on one of them means
nothing on another. "Keep everything above 0.7" is a sentence about monoBERT and only monoBERT;
ColBERT's score of 7.4 is not worse, it is not the same unit, and RankGPT has no number to
threshold at all. The only output all three agree on is the **ordering**, which is why systems
that combine several rerankers combine them over ranks and not over scores.

Three closing rules of thumb:

1. **Measure your first stage before you shop for a reranker.** Stage two can only reorder what
   stage one found. Here stage one had the answer in its top 10 twelve times out of twelve, so a
   reranker could take us to a perfect score. If it had found the answer only 60 percent of the
   time, the best reranker in the world caps out at 60 percent.
2. **A reranker is a budget decision.** monoBERT was accurate and would take hours on a real
   archive. That is why it only ever runs on the shortlist.
3. **Your own numbers or nobody's.** The overfitting in exercise 2 and the position effect in the
   optional cell are both things you have to measure on your own archive, your own questions and
   your own model.

### The four papers

| | |
|---|---|
| Passage Re-ranking with BERT (monoBERT) | Nogueira and Cho, 2019, arXiv 1901.04085 |
| ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction | Khattab and Zaharia, SIGIR 2020, arXiv 2004.12832 |
| Is ChatGPT Good at Search? (RankGPT) | Sun and others, EMNLP 2023, arXiv 2304.09542 |
| Lost in the Middle: How Language Models Use Long Contexts | Liu and others, TACL 2024, arXiv 2307.03172 |

Carlos Cotrini, From Data to Solutions, Weekend 5.
